# x402 payment agent: pay-per-call with Claude

This cookbook shows Claude paying for HTTP 402 endpoints via the
[x402 protocol](https://x402.org) using the [`voidly-pay`](https://pypi.org/project/voidly-pay/)
Python SDK as the wallet + facilitator client.

x402 revives the long-dormant `HTTP 402 Payment Required` status code: the
server returns 402 with a signed quote, the client transfers the asking
amount, and re-issues the call with a `quote_id` (or `X-Payment` header).
Claude needs zero protocol awareness — it just calls a tool and the SDK
handles the dance.

We'll demonstrate four patterns end-to-end against the live
[Voidly Pay rail](https://api.voidly.ai/v1/pay/manifest.json) (settlement
in <200ms via the Sourcify-verified USDC vault on Base mainnet at
[`0xb592...1c12`](https://repo.sourcify.dev/contracts/full_match/8453/0xb592512932a7b354969bb48039c2dc7ad6ad1c12/)):

1. **Inspect a 402 envelope** — see what a paid endpoint actually returns.
2. **Marketplace discovery** — list every paid endpoint on the rail.
3. **Trust report** — verify the vault + chain id before settling.
4. **Settle a real payment** — Claude pays for `/v1/pay/wiki` and
   summarizes the response. Requires a funded DID (see *Bootstrap* below).

> Honest scope note: Voidly Pay is an open marketplace; the rail settles in
> <200ms but real volume is small (~$4 in the vault at the time of writing).
> Treat this as a working pattern for x402 / pay-per-call agent design,
> not a production-grade dependency.


## Setup

Install the Anthropic SDK + `voidly-pay`. The SDK signs every payment with
a local Ed25519 keypair.

In [ ]:
%pip install --upgrade anthropic voidly-pay rich

In [ ]:
import os
import json
import requests
from anthropic import Anthropic
from voidly_pay import VoidlyPay, generate_keypair
from rich import print as rprint

# Set your Anthropic key.
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = Anthropic()
RAIL = "https://api.voidly.ai"

## Bootstrap: get a funded DID

Two ways to get a DID with credits:

### Option A — Browser (zero-install, recommended for first-time use)

Visit **<https://voidly.ai/pay/claim>**. The page mints a DID locally,
calls the faucet for 10 starter credits, and shows you the secret. Set
the values from that page as env vars:

```bash
export VOIDLY_PAY_DID="did:voidly:..."
export VOIDLY_PAY_SECRET="..."  # base64 secret from /pay/claim
```

### Option B — Programmatic (current state of `voidly-pay==1.0.1`)

`generate_keypair()` mints locally; faucet requires the pubkey to be
registered with the rail first. The `/pay/claim` web flow handles that
registration in one click — that's why we recommend Option A for now.
The faucet API path is part of the public manifest; future SDK versions
will register-and-faucet inline.

Once you have credentials, this cell loads them:

In [ ]:
DID = os.environ.get("VOIDLY_PAY_DID")
SECRET = os.environ.get("VOIDLY_PAY_SECRET")

if DID and SECRET:
    pay = VoidlyPay(did=DID, secret_base64=SECRET, api_base=RAIL)
    pay.ensure_wallet()
    rprint(
        {
            "did": pay.did,
            "wallet": pay.wallet(),
            "api_base": pay.api_base,
        }
    )
    HAS_FUNDS = pay.wallet().get("balance_credits", 0) > 0
else:
    pay = None
    HAS_FUNDS = False
    rprint("[yellow]No VOIDLY_PAY_DID set — Demo 4 (settlement) will be skipped.[/yellow]")
    rprint("[yellow]Inspection demos (1-3) still run unchanged.[/yellow]")

## Define Claude's tools

Four tools. The first three need no wallet — they introspect the rail.
The fourth, `voidly_pay_settle`, executes a paid fetch end-to-end.

In [ ]:
TOOLS = [
    {
        "name": "voidly_pay_inspect_402",
        "description": (
            "Make a free GET to a Voidly Pay paid endpoint to retrieve its "
            "402 quote. Returns the price, recipient DID, quote id, and "
            "expiry. Use this BEFORE settling so you know what you're paying."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "Paid endpoint URL."},
            },
            "required": ["url"],
        },
    },
    {
        "name": "voidly_pay_marketplace",
        "description": (
            "List every paid endpoint on the Voidly Pay marketplace. "
            "Returns name, URL, method, price (in USDC and credits)."
        ),
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "voidly_pay_health_check",
        "description": (
            "Trust report for the rail. Returns vault address, chain id, "
            "system_frozen flag, and Sourcify verification link. Always run "
            "this before any high-value settlement."
        ),
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "voidly_pay_settle",
        "description": (
            "Execute a paid fetch end-to-end: inspect the 402 quote, transfer "
            "credits to the recipient DID, then re-call with the quote_id to "
            "get the actual response. Caps at max_amount_credits as a guardrail."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {"type": "string"},
                "max_amount_credits": {"type": "number", "default": 0.01},
            },
            "required": ["url"],
        },
    },
]

## Tool dispatcher

`voidly_pay_settle` shows the full x402 flow:

1. GET the URL → 402 response with `accepts[]`
2. Pick the `voidly-credit` accept option, extract `amount_micro` + `recipient_did` + `quote_id`
3. Sign + submit a `pay()` for that amount to that recipient
4. Re-issue the GET with `?quote_id=<id>` — server verifies the settled transfer and returns the real payload

This is the core loop the [`voidly-pay`](https://pypi.org/project/voidly-pay/)
SDK exposes. Higher-level helpers (`request_with_pay()`) are scheduled for
the next minor.

In [ ]:
def call_tool(name: str, args: dict) -> dict:
    if name == "voidly_pay_inspect_402":
        r = requests.get(args["url"], timeout=15)
        if r.status_code != 402:
            return {
                "status": r.status_code,
                "body": r.text[:1000],
                "note": "Endpoint did not return 402 — it may be free or unavailable.",
            }
        env = r.json()
        # Find the voidly-credit accept option (cleanest agent path).
        credit_accept = next(
            (a for a in env.get("accepts", []) if a.get("scheme") == "voidly-credit"),
            None,
        )
        if not credit_accept:
            return {"error": "no voidly-credit accept option", "envelope": env}
        return {
            "url": args["url"],
            "amount_micro": credit_accept["amount_micro"],
            "amount_usdc": credit_accept["amount_micro"] / 1_000_000,
            "recipient_did": credit_accept["recipient_did"],
            "quote_id": credit_accept["quote_id"],
            "expires_at": credit_accept["expires_at"],
            "description": credit_accept.get("description"),
        }

    if name == "voidly_pay_marketplace":
        return requests.get(f"{RAIL}/v1/pay/marketplace", timeout=15).json()

    if name == "voidly_pay_health_check":
        out = {"checks": []}
        try:
            health = requests.get(f"{RAIL}/v1/pay/health", timeout=10).json()
            out["system_frozen"] = bool(health.get("system_frozen"))
            out["stage"] = health.get("stage")
            out["checks"].append({"name": "api.reachable", "ok": True})
            out["checks"].append(
                {
                    "name": "system.not_frozen",
                    "ok": not out["system_frozen"],
                }
            )
        except Exception as e:
            out["checks"].append({"name": "api.reachable", "ok": False, "hint": str(e)})
        try:
            mfst = requests.get(f"{RAIL}/v1/pay/manifest.json", timeout=10).json()
            bridge = mfst.get("bridge") or {}
            out["vault_address"] = bridge.get("vault_address")
            out["vault_chain_id"] = bridge.get("chain_id")
            out["source_verification"] = bridge.get("source_verification")
            out["checks"].append(
                {
                    "name": "manifest.vault_on_base_mainnet",
                    "ok": bridge.get("chain_id") == 8453,
                }
            )
            out["checks"].append(
                {
                    "name": "manifest.source_verified_link",
                    "ok": bool(bridge.get("source_verification")),
                }
            )
        except Exception as e:
            out["checks"].append({"name": "manifest.fetch", "ok": False, "hint": str(e)})
        out["ok"] = all(c.get("ok") for c in out["checks"])
        return out

    if name == "voidly_pay_settle":
        if pay is None:
            return {
                "skipped": True,
                "reason": "no funded DID — set VOIDLY_PAY_DID + VOIDLY_PAY_SECRET",
                "fix": "https://voidly.ai/pay/claim",
            }
        url = args["url"]
        cap_micro = int(round(args.get("max_amount_credits", 0.01) * 1_000_000))
        # 1) inspect the 402
        r = requests.get(url, timeout=15)
        if r.status_code != 402:
            return {"status": r.status_code, "body": r.text[:600]}
        accepts = r.json().get("accepts", [])
        accept = next((a for a in accepts if a.get("scheme") == "voidly-credit"), None)
        if not accept:
            return {"error": "no voidly-credit option"}
        if accept["amount_micro"] > cap_micro:
            return {
                "blocked_by_cap": True,
                "asked_micro": accept["amount_micro"],
                "cap_micro": cap_micro,
            }
        # 2) settle the transfer
        receipt = pay.pay(
            to=accept["recipient_did"],
            amount_micro=accept["amount_micro"],
            memo=f"x402:{accept['quote_id']}",
        )
        # 3) retry with quote_id
        sep = "&" if "?" in url else "?"
        retry_url = f"{url}{sep}quote_id={accept['quote_id']}"
        retry = requests.get(retry_url, timeout=20)
        return {
            "settled": True,
            "transfer_id": receipt.get("transfer_id"),
            "amount_micro": accept["amount_micro"],
            "recipient": accept["recipient_did"],
            "response_status": retry.status_code,
            "response": retry.json()
            if retry.headers.get("content-type", "").startswith("application/json")
            else retry.text[:1500],
        }

    return {"error": f"unknown tool: {name}"}

## The agent loop

Standard tool-use loop: send the user's message, run any tools Claude
requests, return results, repeat until Claude stops calling tools.

In [ ]:
MODEL = "claude-opus-4-5"  # any tool-capable model works


def run_agent(user_msg: str, max_turns: int = 6) -> str:
    messages = [{"role": "user", "content": user_msg}]
    for turn in range(max_turns):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        tool_results = []
        for block in resp.content:
            if block.type != "tool_use":
                continue
            rprint(f"[bold cyan]→ {block.name}[/bold cyan]({block.input})")
            try:
                out = call_tool(block.name, block.input)
            except Exception as e:
                out = {"error": str(e)}
            rprint(out)
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(out, default=str),
                }
            )
        messages.append({"role": "user", "content": tool_results})
    return "[max turns reached]"

## Demo 1 — Inspect a 402 envelope

No funds needed. Ask Claude to look up the price of the Voidly wiki
endpoint and explain what's in the quote.

In [ ]:
answer = run_agent(
    "Use voidly_pay_inspect_402 on https://api.voidly.ai/v1/pay/wiki and tell me: "
    "(1) what does it cost? (2) who's the recipient? (3) when does the quote expire?"
)
rprint(f"\n[bold green]Claude:[/bold green] {answer}")

## Demo 2 — Marketplace discovery

Let Claude browse every paid endpoint on the rail and pick the cheapest.

In [ ]:
answer = run_agent(
    "List every endpoint on the Voidly Pay marketplace as a markdown table "
    "with: name, what it does (one line), price in USDC, method. Then tell "
    "me which is cheapest."
)
rprint(f"\n[bold green]Claude:[/bold green]\n{answer}")

## Demo 3 — Trust report

Before settling anything real, ask Claude to verify the rail.

In [ ]:
answer = run_agent(
    "Run voidly_pay_health_check. Tell me whether the rail is trustworthy "
    "right now and cite the vault address + Sourcify link."
)
rprint(f"\n[bold green]Claude:[/bold green] {answer}")

## Demo 4 — Settle a real payment (requires funded DID)

If `VOIDLY_PAY_DID` + `VOIDLY_PAY_SECRET` are set, Claude executes the
full x402 flow: inspect → transfer → retry-with-quote-id. Settles in
<200ms.

In [ ]:
if HAS_FUNDS:
    answer = run_agent(
        "Use voidly_pay_settle to fetch https://api.voidly.ai/v1/pay/wiki?topic=Anthropic "
        "(set max_amount_credits=0.005). Then summarize the wiki content in two sentences. "
        "Show me the transfer_id from the receipt."
    )
    rprint(f"\n[bold green]Claude:[/bold green] {answer}")
else:
    rprint("[yellow]Skipping settlement demo — no funded DID.[/yellow]")
    rprint("[yellow]Visit https://voidly.ai/pay/claim to mint one in 30s.[/yellow]")

## What just happened

- Demos 1-3 read the rail freely — no payment needed.
- Demo 4 settled a real payment in <200ms via the on-chain USDC vault.
- Claude never touched a private key — every transfer was Ed25519-signed
  locally by the SDK.
- The settlement produced a receipt; you can list past payments with
  `pay.history(pay.did)`.

### The pattern in 5 lines

```python
quote = requests.get(url).json()["accepts"][0]   # 402
pay.pay(to=quote["recipient_did"], amount_micro=quote["amount_micro"])
data = requests.get(url + f"?quote_id={quote['quote_id']}").json()  # 200
```

Any tool that returns 402 with this envelope shape becomes monetizable.
Any agent with a Voidly Pay wallet becomes a paying client. No platform
lock-in — switch facilitators by changing one URL.

### Where to take this

- **List your own endpoint** at <https://voidly.ai/pay/list-your-service>
  (60s, no review queue).
- **Ship as MCP**: [`@voidly/pay-mcp`](https://www.npmjs.com/package/@voidly/pay-mcp)
  exposes 42 tools to any MCP client (Claude Desktop, Cursor, Claude Code).
- **Other frameworks**:
  [`voidly-pay-langchain`](https://pypi.org/project/voidly-pay-langchain/),
  [`voidly-pay-crewai`](https://pypi.org/project/voidly-pay-crewai/),
  [`voidly-pay-autogen`](https://pypi.org/project/voidly-pay-autogen/) all
  wrap the same primitives.

### Useful links

- Rail manifest: <https://api.voidly.ai/v1/pay/manifest.json>
- Public proof of reserves: <https://voidly.ai/pay/proof>
- x402 spec: <https://x402.org>
- Source-verified vault: <https://repo.sourcify.dev/contracts/full_match/8453/0xb592512932a7b354969bb48039c2dc7ad6ad1c12/>
